# Google client

#### 1. Dependencies

In [1]:
%pip install --quiet google-genai pandas tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: C:\Users\estep\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import time
import pandas as pd
from google import genai
from tqdm import tqdm

#### 2. Configuration

In [3]:
MODEL_NAME = "gemini-2.5-flash"
MODEL_SHORT = "Gemini-2.5-Flash"

def load_api_key(path, name):
    for line in open(path).read().strip().splitlines():
        if line.startswith(f"{name}="):
            return line.split("=", 1)[1].strip()
    raise KeyError(f"Klucz '{name}' nie znaleziony w {path}")

API_KEY = load_api_key("./api_key.txt", "google")

PROMPTS_PATH = "./prompts.tsv"
OUTPUT_DIR = f"./responses_{MODEL_SHORT}"
RESPONSES_PATH = f"{OUTPUT_DIR}/responses.tsv"

POLITE_DELAY = 6.0

TARGET_LANGUAGES = ["zh", "fi", "fr", "he", "ja", "pl"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

#### 3. Load prompts (with resume)

In [4]:
prompts_df = pd.read_csv(PROMPTS_PATH, sep="\t")

try:
    df = pd.read_csv(RESPONSES_PATH, sep="\t")
    df.loc[df["response"].astype(str).str.startswith("EXCEPTION:"), "response"] = pd.NA
    done = df["response"].notna().sum()
    print(f"Wznowiono: {done}/{len(df)} wykonanych")
except FileNotFoundError:
    df = prompts_df.copy()
    df["response"] = pd.NA
    print(f"Start od poczatku: {len(df)} promptow do wyslania")

Wznowiono: 19/1422 wykonanych


#### 4. Send loop

In [5]:
client = genai.Client(api_key=API_KEY)

MAX_RETRIES = 4
RETRY_BASE_DELAY = 15.0

def is_transient(exc):
    msg = str(exc).upper()
    return any(kw in msg for kw in ("503", "UNAVAILABLE", "OVERLOAD", "TRY AGAIN", "TIMEOUT"))

def is_daily_quota(exc):
    """Daily/per-project quota - retry nie pomoze, trzeba czekac do polnocy UTC."""
    msg = str(exc).upper()
    return ("GENERATEREQUESTSPERDAY" in msg or
            ("RESOURCE_EXHAUSTED" in msg and "PERDAY" in msg) or
            ("QUOTA" in msg and "DAY" in msg))

class DailyQuotaError(Exception):
    pass

def send_one(prompt):
    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
            return response.text or ""
        except Exception as exc:
            if is_daily_quota(exc):
                raise DailyQuotaError(str(exc))
            if is_transient(exc) and attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_BASE_DELAY * (2 ** attempt))
            else:
                raise

pending = df[df["response"].isna()].index.tolist()
print(f"Do wyslania: {len(pending)} (delay={POLITE_DELAY}s)")

failed = 0
quota_skipped = 0
quota_exhausted = False

for i in tqdm(pending, desc="Wysylanie"):
    if quota_exhausted:
        df.at[i, "response"] = pd.NA
        quota_skipped += 1
        continue
    time.sleep(POLITE_DELAY)
    try:
        df.at[i, "response"] = send_one(df.at[i, "prompt"])
    except DailyQuotaError:
        quota_exhausted = True
        df.at[i, "response"] = pd.NA
        quota_skipped += 1
    except Exception:
        df.at[i, "response"] = pd.NA
        failed += 1
    df.to_csv(RESPONSES_PATH, sep="\t", index=False)

done = len(pending) - failed - quota_skipped
print(f"Sukces: {done}, blad: {failed}, daily_quota_skip: {quota_skipped}")
if quota_exhausted:
    print("Daily quota wyczerpana. Wroc jutro (reset o polnocy UTC = 1:00/2:00 czasu polskiego).")
    print("Notebook automatycznie wznowi od miejsca przerwania.")

Do wyslania: 1403 (delay=6.0s)


Wysylanie: 100%|██████████| 1403/1403 [00:07<00:00, 198.29it/s]

Sukces: 0, blad: 0, daily_quota_skip: 1403
Daily quota wyczerpana. Wroc jutro (reset o polnocy UTC = 1:00/2:00 czasu polskiego).
Notebook automatycznie wznowi od miejsca przerwania.
